In [0]:
df=spark.read.format('parquet')\
    .option('inferschema','true')\
    .option('header','true')\
    .load('/Volumes/catalog/bronze/bronze/raw_bronze/')

In [0]:
display(df.limit(5))

Branch_ID,Dealer_ID,Model_ID,Revenue,Units_Sold,Date_ID,Day,Month,Year,BranchName,DealerName,Product_Name
BR0001,DLR0001,BMW-M1,13363978,2,DT00001,1,1,2017,AC Cars Motors,AC Cars Motors,BMW
BR0003,DLR0228,Hon-M218,17376468,3,DT00001,10,5,2017,AC Cars Motors,Deccan Motors,Honda
BR0004,DLR0208,Tat-M188,9664767,3,DT00002,12,1,2017,AC Cars Motors,Wiesmann Motors,Tata
BR0005,DLR0188,Hyu-M158,5525304,3,DT00002,16,9,2017,AC Cars Motors,Subaru Motors,Hyundai
BR0006,DLR0168,Ren-M128,12971088,3,DT00003,20,5,2017,AC Cars Motors,Saab Motors,Renault


In [0]:
df_branch=spark.sql('''select * from catalog.gold.dim_branch ''')
df_date=spark.sql('''select * from catalog.gold.dim_date ''')
df_dealer=spark.sql('''select * from catalog.gold.dim_dealer ''')
df_model= spark.sql('''select * from catalog.gold.dim_model ''')


In [0]:
df_fact=df.join(df_branch,df['Branch_ID']==df_branch['Branch_ID'],'inner')\
    .join(df_date,df['Date_ID']==df_date['Date_ID'],'inner')\
    .join(df_dealer,df['Dealer_ID']==df_dealer['Dealer_ID'],'inner')\
    .join(df_model,df['Model_ID']==df_model['Model_ID'],'inner')\
    .select(df.Branch_ID,df.Date_ID,df.Dealer_ID,df.Model_ID,df.Revenue,df.Units_Sold,df_date.dim_date_key,df_branch.dim_branch_key,df_dealer.dim_dealer_key,df_model.dim_model_key)


In [0]:
df_fact.limit(5).display()

Branch_ID,Date_ID,Dealer_ID,Model_ID,Revenue,Units_Sold,dim_date_key,dim_branch_key,dim_dealer_key,dim_model_key
BR0004,DT00002,DLR0208,Tat-M188,9664767,3,866,1,67,148
BR0012,DT00006,DLR0249,BMW-M249,5358057,1,1001,2,2,226
BR0013,DT00007,DLR0229,Hon-M219,16150431,3,434,3,199,1
BR0015,DT00008,DLR0189,Hyu-M159,4891618,2,70,4,4,39
BR0051,DT00031,DLR0073,Vol-M260,2224368,1,734,5,148,228


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("catalog.gold.fact_table"):
    delta_table=DeltaTable.forPath('spark','/Volumes/catalog/gold/fact_table')
    delta_table.alias('t').merge(df_fact.alias('s'),'t.Branch_ID=s.Branch_ID and t.Date_ID=s.Date_ID and t.Dealer_ID=s.Dealer_ID and t.Model_ID=s.Model_ID')\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()
    .execute()


else:
    df_fact.write.format('Delta').mode('overwrite').saveAsTable('catalog.gold.fact_table')
    df_fact.write.format('Delta').mode('overwrite').save('/Volumes/catalog/gold/gold/fact_table')
